In [1]:
#import library
from pandas_gbq import read_gbq
import pandas as pd
import numpy as np
import os
import datetime
import ssl
import logging

In [2]:
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

# Get today's date
today = date.today()

# --- Current month period (CY): last completed calendar month ---
# last day of previous month
end_date = date(today.year, today.month, 1) - timedelta(days=1)
# first day of that month
start_date = date(end_date.year, end_date.month, 1)

# --- Prior period (PP): previous calendar month (1 month) ---
pp_end_date = start_date - timedelta(days=1)
pp_start_date = date(pp_end_date.year, pp_end_date.month, 1)

# --- Corresponding periods last year (LY): same calendar months ---
# "CWLY" = same month last year (range)
cwly_start_date = start_date - relativedelta(years=1)
cwly_end_date   = end_date   - relativedelta(years=1)

# "PWLY" = prior-month period last year (range)
pwly_start_date = pp_start_date - relativedelta(years=1)
pwly_end_date   = pp_end_date   - relativedelta(years=1)

# --- YTD helpers ---
ytd_last_year = end_date - relativedelta(years=1)  # same day last year as this period end
begin_of_current_year = date(end_date.year, 1, 1)
begin_of_last_year    = date(end_date.year - 1, 1, 1)

# Output all dates (I return ranges for LY to match month logic)
(
    end_date, start_date,
    pp_end_date, pp_start_date,
    cwly_start_date, cwly_end_date,
    pwly_start_date, pwly_end_date,
    begin_of_last_year, ytd_last_year, begin_of_current_year
)


(datetime.date(2026, 1, 31),
 datetime.date(2026, 1, 1),
 datetime.date(2025, 12, 31),
 datetime.date(2025, 12, 1),
 datetime.date(2025, 1, 1),
 datetime.date(2025, 1, 31),
 datetime.date(2024, 12, 1),
 datetime.date(2024, 12, 31),
 datetime.date(2025, 1, 1),
 datetime.date(2025, 1, 31),
 datetime.date(2026, 1, 1))

In [3]:
from custom_query import read_sql_query, sql_files

# Read SQL queries from files
queries = {key: read_sql_query(path) for key, path in sql_files.items()}

# Execute queries if all were successfully reada
if all(queries.values()):
    try:
        monthly_note = read_gbq(queries["weekly_note"], project_id='pcln-pl-airanalytics-prod')
        finance_data = read_gbq(queries["finance_data"], project_id='pcln-pl-airanalytics-prod')
        gds_incentives = read_gbq(queries["gds_incentives"], project_id='pcln-pl-airanalytics-prod')
        tsa_data = read_gbq(queries["tsa_data"], project_id='pcln-pl-airanalytics-prod')
        dau_conversion_data = read_gbq(queries["dau_conversion_query"], project_id='pcln-pl-airanalytics-prod')
        print("SQL queries executed successfully.")
        
    except Exception as e:
        print(f"Failed to execute SQL queries: {e}")

/Users/sye/Library/Python/3.9/lib/python/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=262006177488-3425ks60hkk80fssi9vpohv88g6q1iqd.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&state=NQKsBQZ1eYvuKfdi6u9GXEjdDXFh8W&prompt=consent&access_type=offline
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
SQL queries executed successfully.


In [4]:
# make a copy of every data set
df_monthly=monthly_note.copy()
df_finance=finance_data.copy()
df_gds_incentive=gds_incentives.copy()
df_tsa=tsa_data.copy()
df_monthly.columns = df_monthly.columns.str.lower()
dau_conversion=dau_conversion_data.copy()

## Summary Table

In [33]:

def format_number(num):
    if pd.isna(num):
        return ''
    if abs(num) >= 1e6:
        return f"{num/1e6:.1f}M"
    elif abs(num) >= 1e3:
        return f"{num/1e3:.0f}K"
    elif abs(num) < 1e3:
        return "<1K"
    return f"{num:.0f}"


def format_percentage(x, decimals=1, multiply_100=True):
    def fmt(v):
        if pd.isna(v):
            return ''
        v = float(v)
        if multiply_100:
            v *= 1
        return f"{v:.{decimals}f}%"

    if isinstance(x, pd.Series):
        return x.map(fmt)
    if isinstance(x, pd.DataFrame):
        return x.applymap(fmt) 
    return fmt(x)  


def format_percentage_2(num):
    if pd.isna(num):
        return ''
    return f"{num:.2f}%"

def round_to_nearest_10(num):
  return round(num / 10) * 10

df_pricelince=df_monthly[(df_monthly['brand']== 'Priceline')]
df_pricelince_air=df_monthly[(df_monthly['brand']== 'Priceline')&(df_monthly['offer_type']== 'Flights Only')]
df_pricelince_b2c=df_monthly[(df_monthly['brand']== 'Priceline')&(df_monthly['company']== 'Priceline B2C')]
df_pricelince_b2c_standalone=df_monthly[(df_monthly['brand']== 'Priceline')&(df_monthly['offer_type']== 'Flights Only')&(df_monthly['company']== 'Priceline B2C')]


In [32]:
df_pricelince_b2c_standalone

,trans_date,us_travel_type,company,brand,offer_type,carrier_detail,carrier,offer_method_code,search_channel,search_channel_group,...,net_contr_fee_cy,yoy_net_contr_fee,net_orders_cy,net_orders_ly,gr_wex_fee_cy,gr_wex_fee_ly,ref_wex_fee_cy,ref_wex_fee_ly,net_wex_fee_cy,net_wex_fee_ly
349,2024-12-01,Domestic,Priceline B2C,Priceline,Flights Only,ANA All Nippon Airways (NH),Other,Retail (Disclosed),SEM Brand,Web Marketing,...,9.96,-9.96,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
350,2024-12-01,Domestic,Priceline B2C,Priceline,Flights Only,Advanced Air (AN),Other,Retail (Disclosed),Commission Junction,Affiliate,...,0.00,0.00,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
351,2024-12-01,Domestic,Priceline B2C,Priceline,Flights Only,Advanced Air (AN),Other,Retail (Disclosed),Email,Direct,...,0.00,0.00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
352,2024-12-01,Domestic,Priceline B2C,Priceline,Flights Only,Advanced Air (AN),Other,Retail (Disclosed),Meta,Shop PPC,...,0.00,0.00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
353,2024-12-01,Domestic,Priceline B2C,Priceline,Flights Only,Advanced Air (AN),Other,Retail (Disclosed),Meta,Shop PPC,...,0.00,3.10,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
821535,2026-02-01,X,Priceline B2C,Priceline,Packages,avianca (AV),Other,Retail (Disclosed),Site.com,Direct,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
821536,2026-02-01,X,Priceline B2C,Priceline,Packages,avianca (AV),Other,Retail (Disclosed),Site.com,Direct,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
821537,2026-02-01,X,Priceline B2C,Priceline,Packages,easyJet (U2),Other,Retail (Disclosed),Commission Junction,Affiliate,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
821538,2026-02-01,X,Priceline B2C,Priceline,Packages,interCaribbean (JY),Other,Retail (Disclosed),SEM Core,Web Marketing,...,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
import kpi_month
import importlib

importlib.reload(kpi_month)

from kpi_month import calculate_business_metrics

# Calculate business metrics
df_business = calculate_business_metrics(
    df_pricelince,
    start_date,
    end_date, 
    pp_start_date,
    pp_end_date, 
    cwly_start_date, 
    cwly_end_date,       
    pwly_start_date, 
    pwly_end_date, 
    format_number)


from kpi_month import calculate_carrier_metrics
# Calculate carrier metrics
df_carrier = calculate_carrier_metrics(
    df_pricelince_b2c_standalone,
    start_date,
    end_date,
    pp_start_date,
    pp_end_date,
    cwly_start_date, 
    cwly_end_date,       
    pwly_start_date, 
    pwly_end_date,   
    format_number,
)


from kpi_month import calculate_channel_metrics
df_channel = calculate_channel_metrics(
    df_pricelince_b2c_standalone,
    start_date, end_date,
    pp_start_date, pp_end_date,
    cwly_start_date, 
    cwly_end_date,       
    pwly_start_date, 
    pwly_end_date, 
    format_number,
)

In [35]:
df_business

,Net Tickets_standalone,YoY_standalone,YoY PW_standalone,Net Tickets_package,YoY_package,YoY PW_package,Net Tickets_total,YoY_total,YoY PW_total
B2C,631K,-2.5%,-5.7%,147K,27.4%,25.9%,778K,2.1%,-1.9%
B2B,26K,-18.2%,-10.0%,28K,-1.9%,3.0%,54K,-10.5%,-5.1%
Total,657K,-3.2%,-5.9%,175K,21.6%,21.9%,832K,1.1%,-2.1%


In [36]:
df_channel

,Net Tickets_App,YoY_App,YoY PW_App,Net Tickets_Desk/MWEB,YoY_Desk/MWEB,YoY PW_Desk/MWEB,Net Tickets_Total,YoY_Total,YoY PW_Total
Direct,134K,-0.5%,-6.3%,89K,-24.8%,-23.7%,223K,-11.8%,-14.2%
Web Marketing,37K,112.9%,99.3%,231K,1.8%,-2.7%,268K,9.8%,4.8%
Shop PPC,25K,-10.3%,-10.7%,73K,-5.6%,-6.8%,98K,-6.8%,-7.8%
Affiliate,1K,290.3%,157.0%,40K,-8.0%,-12.1%,42K,-6.0%,-10.6%
Total,198K,9.5%,3.6%,433K,-7.1%,-9.5%,631K,-2.5%,-5.7%


In [37]:
df_carrier

,Net Tickets_Retail,YoY_Retail,YoY PW_Retail,Net Tickets_Opaque,YoY_Opaque,YoY PW_Opaque,Net Tickets_Total,YoY_Total,YoY PW_Total
American Airlines (AA),109K,-16.8%,-18.5%,6K,-82.58797313608368%,-78.33182230281052%,115K,-29.8%,-31.9%
Delta Air Lines (DL),79K,-16.7%,-20.2%,<1K,-89.38329430132708%,-67.04302715898687%,79K,-21.3%,-21.9%
United Airlines (UA),62K,-11.2%,-10.1%,39K,1.4279758182197178%,27.888237756383425%,101K,-6.7%,1.1%
Southwest Airlines (WN),75K,20614.9%,18856.7%,<1K,50300.0%,<NA>%,76K,20696.4%,19180.1%
Spirit Airlines (NK),37K,-46.6%,-48.7%,,,,37K,-46.6%,-48.7%
Frontier Airlines (F9),78K,34.5%,16.8%,<1K,<NA>%,<NA>%,78K,34.5%,16.8%
Alaska Airlines (AS),23K,11.7%,-26.6%,19K,49.808429118773944%,54.49568525563699%,41K,26.3%,-0.4%
JetBlue Airways (B6),26K,-2.4%,10.8%,<1K,7000.0%,<NA>%,27K,-2.2%,11.1%
Other,76K,-11.5%,-15.7%,<1K,551.2820512820513%,528.0991735537191%,76K,-10.7%,-14.9%
Total,565K,1.5%,-3.6%,65K,-27.19441937173358%,-19.773476880950902%,631K,-2.5%,-5.7%


##  DAU


In [10]:
import importlib
import dau_roi_table_month

importlib.reload(dau_roi_table_month)

from dau_roi_table_month import calculate_dau_conversion

df_dau_conversion = calculate_dau_conversion(
    dau_conversion,
    format_percentage,
    start_date,
    end_date,
    pp_start_date,
    pp_end_date,
    cwly_start_date,
    cwly_end_date,
    pwly_start_date,
    pwly_end_date,
    begin_of_current_year,
    begin_of_last_year,
    ytd_last_year)

df_dau_conversion

,DAU,DAU_pw,DAU_cwly,DAU_pwly,DAU_ytd,DAU_ytd_ly,DAU YoY,DAU YoY_PW,DAU YoY_YTD
channel,,,,,,,,,
Affiliate,53726,56912,29825,27682,53726,29825,80.1%,105.6%,80.1%
Direct,3560842,3361849,4023438,3659708,3560842,4023438,-11.5%,-8.1%,-11.5%
SEM Brand,596012,561502,704199,627969,596012,704199,-15.4%,-10.6%,-15.4%
SEM Core,2103797,2058086,1967059,1812714,2103797,1967059,7.0%,13.5%,7.0%
Shop PPC Cheapflights,908840,1118639,1137616,1208905,908840,1137616,-20.1%,-7.5%,-20.1%
Shop PPC Google,2219,1977,2244,1871,2219,2244,-1.1%,5.7%,-1.1%
Shop PPC Kayak,634509,576759,214811,172335,634509,214811,195.4%,234.7%,195.4%
Shop PPC Others,539329,515193,577391,480846,539329,577391,-6.6%,7.1%,-6.6%
Total,8399274,8250917,8656583,7992030,8399274,8656583,-3.0%,3.2%,-3.0%


# Summary Table

## YTD

In [11]:

import importlib
import summary_table_actual_month  

importlib.reload(summary_table_actual_month)

from summary_table_actual_month import create_finance_number_month


finance_number = create_finance_number_month(
    df_monthly,
    df_gds_incentive,
    start_date, end_date,                 
    pp_start_date, pp_end_date,           
    cwly_start_date, cwly_end_date,       
    pwly_start_date, pwly_end_date,       
    begin_of_current_year,
    begin_of_last_year,
    ytd_last_year,
    format_percentage,
    format_number,
)

finance_number

,Measure,CW,PW,CWly,pWly,Reporting Week,Previous Week,CY,LY,YTD
0,Net Tickets,831730.0,731457.0,822325.0,747287.0,1.143708,-2.118329,831730.0,822325.0,1.143708
1,Gross Tickets,894175.0,788225.0,886214.0,802249.0,0.898316,-1.748086,894175.0,886214.0,0.898316
2,Net Revenue (net_contribution),8332959.0,7619293.0,9719182.0,8198923.0,-14.262749,-7.069593,8332959.0,9719182.0,-14.262749
3,Gross Revenue (gross_contribution),9057984.0,8308281.0,10749206.0,8934944.0,-15.733459,-7.013612,9057984.0,10749206.0,-15.733459
4,Normalized Net Tickets,717653.0,630875.0,710103.0,646828.0,1.063226,-2.466343,717653.0,710103.0,1.063226
5,Normalized Gross Tickets,769502.0,677770.0,766187.0,693989.0,0.432662,-2.337069,769502.0,766187.0,0.432662
6,Net Cont + Fee,9455709.0,8939144.0,10515782.0,8978417.0,-10.080779,-0.437410,9455709.0,10515782.0,-10.080779
7,Gross Cont + Fee,10254049.0,9713065.0,11610098.0,9784890.0,-11.679909,-0.734033,10254049.0,11610098.0,-11.679909
8,GDS Incentive,482876.0,419605.0,866571.0,792301.0,-44.277434,-47.039733,482876.0,866571.0,-44.277434
9,VCC Rebate (net_wex_fee),389512.0,435382.0,131566.0,153467.0,196.058805,183.697906,389512.0,131566.0,196.058805


In [12]:
#current week finance data
#net tickets by scenario
plan_net_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_cw=df_finance[df_finance['year_month']==start_date.strftime('%Y-%m-%d')].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)

#previous week finance data
#net tickets by scenario
plan_net_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_pw=df_finance[df_finance['year_month']==pp_start_date.strftime('%Y-%m-%d')].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)


#ytd week finance data
#net tickets by scenario
plan_net_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_ytd=df_finance[(df_finance['year_month']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)



In [13]:
period_order = ["CW", "PW", "YTD", "YTD_LY"]

# revenue as Series (no squeeze)
plan_grrev_cw     = df_finance.loc[df_finance["year_month"] == start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_pw     = df_finance.loc[df_finance["year_month"] == pp_start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_ytd    = df_finance.loc[
    (df_finance["year_month"] <= end_date.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_current_year.strftime("%Y-%m-%d"))
].groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_ytd_ly = df_finance.loc[
    (df_finance["year_month"] <= ytd_last_year.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_last_year.strftime("%Y-%m-%d"))
].groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_cw     = df_finance.loc[df_finance["year_month"] == start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_pw     = df_finance.loc[df_finance["year_month"] == pp_start_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_ytd    = df_finance.loc[
    (df_finance["year_month"] <= start_date.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_current_year.strftime("%Y-%m-%d"))
].groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_ytd_ly = df_finance.loc[
    (df_finance["year_month"] <= ytd_last_year.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_last_year.strftime("%Y-%m-%d"))
].groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

# 1) MultiIndex columns (metric, period)
plan_metrics_mi = pd.concat(
    {
        ("net_tkts", "CW"): plan_net_cw,
        ("net_tkts", "PW"): plan_net_pw,
        ("net_tkts", "YTD"): plan_net_ytd,
        # ("net_tkts", "YTD_LY"): plan_net_ytd_ly,

        ("gr_tkts", "CW"): plan_gr_cw,
        ("gr_tkts", "PW"): plan_gr_pw,
        ("gr_tkts", "YTD"): plan_gr_ytd,
        # ("gr_tkts", "YTD_LY"): plan_gr_ytd_ly,

        ("gr_rev", "CW"): plan_grrev_cw,
        ("gr_rev", "PW"): plan_grrev_pw,
        ("gr_rev", "YTD"): plan_grrev_ytd,
        # ("gr_rev", "YTD_LY"): plan_grrev_ytd_ly,

        ("net_rev", "CW"): plan_netrev_cw,
        ("net_rev", "PW"): plan_netrev_pw,
        ("net_rev", "YTD"): plan_netrev_ytd,
        # ("net_rev", "YTD_LY"): plan_netrev_ytd_ly,
    },
    axis=1
)
plan_metrics_mi.index.name = "scenario"

# 2) reshape to "period columns"
plan_metrics_period_cols = (
    plan_metrics_mi
      .stack(0)                # stack metric level -> rows
      .reset_index()
      .rename(columns={"level_1": "metric"})
)

# optional formatting
if callable(format_number):
    for c in period_order:
        if c in plan_metrics_period_cols.columns:
            plan_metrics_period_cols[c] = plan_metrics_period_cols[c].round(0)

keep = ["scenario", "metric"] + [c for c in period_order if c in plan_metrics_period_cols.columns]
plan_metrics_period = plan_metrics_period_cols[keep]

# filter PLAN
plan_metrics_period_plan = plan_metrics_period.loc[plan_metrics_period["scenario"].eq("PLAN")]
plan_metrics_period_plan

/var/folders/k6/ygnww5yd0rx_9c55x7xb7xs00000gn/T/ipykernel_81689/3762684958.py:65: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  plan_metrics_mi


,scenario,metric,CW,PW,YTD
0,PLAN,gr_rev,8909896.0,9547312.0,8909896.0
1,PLAN,gr_tkts,883057.0,787376.0,883057.0
2,PLAN,net_rev,8423038.0,8910291.0,8423038.0
3,PLAN,net_tkts,830837.0,731953.0,830837.0


In [14]:
# import importlib
# import build_plan_metrics
# importlib.reload(build_plan_metrics)
# from build_plan_metrics import build_plan_metrics_period_plan



# plan_metrics_period_plan =  build_plan_metrics_period_plan(
#     df_finance,
#     start_date,
#     end_date,
#     pp_start_date,
#     begin_of_current_year,
#     begin_of_last_year,
#     ytd_last_year,
#     format_number=None,
#     period_order=None,
#     scenario_value="PLAN",
# )

In [15]:

import importlib
import summary_table_actual_month  

importlib.reload(summary_table_actual_month)

from summary_table_actual_month import create_finance_number_month


finance_number = create_finance_number_month(
    df_monthly,
    df_gds_incentive,
    start_date, end_date,                 
    pp_start_date, pp_end_date,           
    cwly_start_date, cwly_end_date,       
    pwly_start_date, pwly_end_date,       
    begin_of_current_year,
    begin_of_last_year,
    ytd_last_year,
    format_percentage,
    format_number,
)

finance_number

,Measure,CW,PW,CWly,pWly,Reporting Week,Previous Week,CY,LY,YTD
0,Net Tickets,831730.0,731457.0,822325.0,747287.0,1.143708,-2.118329,831730.0,822325.0,1.143708
1,Gross Tickets,894175.0,788225.0,886214.0,802249.0,0.898316,-1.748086,894175.0,886214.0,0.898316
2,Net Revenue (net_contribution),8332959.0,7619293.0,9719182.0,8198923.0,-14.262749,-7.069593,8332959.0,9719182.0,-14.262749
3,Gross Revenue (gross_contribution),9057984.0,8308281.0,10749206.0,8934944.0,-15.733459,-7.013612,9057984.0,10749206.0,-15.733459
4,Normalized Net Tickets,717653.0,630875.0,710103.0,646828.0,1.063226,-2.466343,717653.0,710103.0,1.063226
5,Normalized Gross Tickets,769502.0,677770.0,766187.0,693989.0,0.432662,-2.337069,769502.0,766187.0,0.432662
6,Net Cont + Fee,9455709.0,8939144.0,10515782.0,8978417.0,-10.080779,-0.437410,9455709.0,10515782.0,-10.080779
7,Gross Cont + Fee,10254049.0,9713065.0,11610098.0,9784890.0,-11.679909,-0.734033,10254049.0,11610098.0,-11.679909
8,GDS Incentive,482876.0,419605.0,866571.0,792301.0,-44.277434,-47.039733,482876.0,866571.0,-44.277434
9,VCC Rebate (net_wex_fee),389512.0,435382.0,131566.0,153467.0,196.058805,183.697906,389512.0,131566.0,196.058805


In [16]:
def build_vs_plan(finance_number: pd.DataFrame, plan_metrics_period: pd.DataFrame,format_percentage) -> pd.DataFrame:
    plan = (plan_metrics_period.loc[plan_metrics_period["scenario"].eq("PLAN"),
                                    ["metric", "CW", "PW", "YTD"]]
            .set_index("metric")
            .rename(columns={"CW": "CW_plan", "PW": "PW_plan", "YTD": "YTD_plan"}))

    actual = (finance_number[["Measure", "CW", "PW", "CY"]]
              .set_index("Measure"))

    measure_to_metric = {
        "Net Tickets": "net_tkts",
        "Gross Tickets": "gr_tkts",
        "Net Cont + Fee + Incentives + vcc rebate(Flight Only)": "net_rev",
        # "Gross Cont + Fee + Incentives + vcc rebate(Flight Only)": "gr_rev",
    }

    tmp = actual.rename(index=measure_to_metric)
    tmp = tmp.join(plan, how="right")
    
    metric_order = list(measure_to_metric.values())
    tmp = tmp.reindex(metric_order)
    def vs_pct(a, p):
        p = p.replace(0, np.nan)
        return (a / p - 1) * 100

    out = pd.DataFrame(index=tmp.index)
    out["Reporting Week (vs Plan)"] = vs_pct(tmp["CW"], tmp["CW_plan"])
    out["Previous Week (vs Plan)"]  = vs_pct(tmp["PW"], tmp["PW_plan"])
    out["YTD (vs Plan)"]            = vs_pct(tmp["CY"], tmp["YTD_plan"])  # CY = YTD actual

    metric_to_measure = {v: k for k, v in measure_to_metric.items()}
    out.index = out.index.map(metric_to_measure)

    for c in ["Reporting Week (vs Plan)", "Previous Week (vs Plan)", "YTD (vs Plan)"]:
        out[c] = out[c].apply(format_percentage)

    return out



In [17]:
finance_number

,Measure,CW,PW,CWly,pWly,Reporting Week,Previous Week,CY,LY,YTD
0,Net Tickets,831730.0,731457.0,822325.0,747287.0,1.143708,-2.118329,831730.0,822325.0,1.143708
1,Gross Tickets,894175.0,788225.0,886214.0,802249.0,0.898316,-1.748086,894175.0,886214.0,0.898316
2,Net Revenue (net_contribution),8332959.0,7619293.0,9719182.0,8198923.0,-14.262749,-7.069593,8332959.0,9719182.0,-14.262749
3,Gross Revenue (gross_contribution),9057984.0,8308281.0,10749206.0,8934944.0,-15.733459,-7.013612,9057984.0,10749206.0,-15.733459
4,Normalized Net Tickets,717653.0,630875.0,710103.0,646828.0,1.063226,-2.466343,717653.0,710103.0,1.063226
5,Normalized Gross Tickets,769502.0,677770.0,766187.0,693989.0,0.432662,-2.337069,769502.0,766187.0,0.432662
6,Net Cont + Fee,9455709.0,8939144.0,10515782.0,8978417.0,-10.080779,-0.437410,9455709.0,10515782.0,-10.080779
7,Gross Cont + Fee,10254049.0,9713065.0,11610098.0,9784890.0,-11.679909,-0.734033,10254049.0,11610098.0,-11.679909
8,GDS Incentive,482876.0,419605.0,866571.0,792301.0,-44.277434,-47.039733,482876.0,866571.0,-44.277434
9,VCC Rebate (net_wex_fee),389512.0,435382.0,131566.0,153467.0,196.058805,183.697906,389512.0,131566.0,196.058805


In [18]:
plan_metrics_period

,scenario,metric,CW,PW,YTD
0,PLAN,gr_rev,8909896.0,9547312.0,8909896.0
1,PLAN,gr_tkts,883057.0,787376.0,883057.0
2,PLAN,net_rev,8423038.0,8910291.0,8423038.0
3,PLAN,net_tkts,830837.0,731953.0,830837.0


In [19]:
df_vs_plan = build_vs_plan(finance_number, plan_metrics_period, format_percentage)
df_vs_plan

,Reporting Week (vs Plan),Previous Week (vs Plan),YTD (vs Plan)
metric,,,
Net Tickets,0.1%,-0.1%,0.1%
Gross Tickets,1.3%,0.1%,1.3%
Net Cont + Fee + Incentives + vcc rebate(Flight Only),0.1%,-5.6%,0.1%


In [20]:
finance_number
wanted = [
    "Net Tickets",
    "Gross Tickets",
    "Net Cont + Fee + Incentives + vcc rebate(Flight Only)",
    "Gross Cont + Fee + Incentives + vcc rebate(Flight Only)",
    "Normalized Net Tickets",
    "Normalized Gross Tickets",
]

finance_output = finance_number.loc[
    finance_number["Measure"].isin(wanted),
    ["Measure", "CW", "Reporting Week", "Previous Week", "YTD"]
].reset_index(drop=True)

# ---- formatting ----
num_cols = ["CW"]
pct_cols = ["Reporting Week", "Previous Week", "YTD"]

if callable(format_number):
    for c in num_cols:
        finance_output[c] = finance_output[c].apply(format_number)

if callable(format_percentage):
    for c in pct_cols:
        finance_output[c] = finance_output[c].apply(format_percentage)

finance_output=finance_output.set_index('Measure')
finance_output=finance_output.rename(columns={"CW": "Actual"})
finance_output=finance_output.reindex(wanted).fillna('')
finance_output


,Actual,Reporting Week,Previous Week,YTD
Measure,,,,
Net Tickets,832K,1.1%,-2.1%,1.1%
Gross Tickets,894K,0.9%,-1.7%,0.9%
Net Cont + Fee + Incentives + vcc rebate(Flight Only),8.4M,-11.7%,-2.0%,-11.7%
Gross Cont + Fee + Incentives + vcc rebate(Flight Only),9.1M,-13.0%,-1.8%,-13.0%
Normalized Net Tickets,718K,1.1%,-2.5%,1.1%
Normalized Gross Tickets,770K,0.4%,-2.3%,0.4%


## TSA

In [39]:
tsa_cy=df_tsa[(df_tsa['date']>=start_date)&(df_tsa['date']<=end_date)]['tsa_passengers'].sum()
pcln_cy=df_tsa[(df_tsa['date']>=start_date)&(df_tsa['date']<=end_date)]['pcln_passengers'].sum()
tsa_actual_cy=(pcln_cy*100/tsa_cy)

tsa_pw=df_tsa[(df_tsa['date']>=pp_start_date)&(df_tsa['date']<=pp_end_date)]['tsa_passengers'].sum()
pcln_pw=df_tsa[(df_tsa['date']>=pp_start_date)&(df_tsa['date']<=pp_end_date)]['pcln_passengers'].sum()
tsa_actual_pw=(pcln_pw*100/tsa_pw)

tsa_cwly=df_tsa[(df_tsa['date']>=cwly_start_date)&(df_tsa['date']<=cwly_end_date)]['tsa_passengers'].sum()
pcln_cwly=df_tsa[(df_tsa['date']>=cwly_start_date)&(df_tsa['date']<=cwly_end_date)]['pcln_passengers'].sum()
tsa_actual_cwly=(pcln_cwly*100/tsa_cwly)


tsa_pwly=df_tsa[(df_tsa['date']>=pwly_start_date)&(df_tsa['date']<=pwly_end_date)]['tsa_passengers'].sum()
pcln_pwly=df_tsa[(df_tsa['date']>=pwly_start_date)&(df_tsa['date']<=pwly_end_date)]['pcln_passengers'].sum()
tsa_actual_pwly=(pcln_pwly*100/tsa_pwly)


tsa_ytd=df_tsa[(df_tsa['date'] <=end_date)&(df_tsa['date']>=begin_of_current_year)]['tsa_passengers'].sum()
pcln_ytd=df_tsa[(df_tsa['date']<=end_date)& (df_tsa['date']>=begin_of_current_year)]['pcln_passengers'].sum()
tsa_actual_ytd=round((pcln_ytd*100/tsa_ytd),2)


tsa_ytd_ly=df_tsa[(df_tsa['date'] <=ytd_last_year)&(df_tsa['date']>=begin_of_last_year)]['tsa_passengers'].sum()
pcln_ytd_ly=df_tsa[(df_tsa['date']<=ytd_last_year)& (df_tsa['date']>=begin_of_last_year)]['pcln_passengers'].sum()
tsa_actual_ytd_ly=round((pcln_ytd_ly*100/tsa_ytd_ly),2)

print("TSA CY",tsa_cy,"PCLN CY",pcln_cy,"%TSA market share",tsa_actual_cy)
print("TSA LY",tsa_cwly,"PCLN LY",pcln_cwly,"%TSAmarket share",tsa_actual_cwly)
print("TSA PW",tsa_pw,"PCLN PW",pcln_pw,"%TSAmarket share",tsa_actual_pw)
print("TSA PWLY",tsa_pwly,"PCLN PWly",pcln_pwly,"%TSAmarket share",tsa_actual_pwly)
print("TSA YTD CY",tsa_ytd,"PCLN YTD CY",pcln_ytd,"%TSAmarket share CY YTD ",tsa_actual_ytd)
print("TSA YTD LY",tsa_ytd_ly,"PCLN YTD LY",pcln_cwly,"%TSA market share LY YTD ",tsa_actual_ytd_ly)

TSA CY 65860911 PCLN CY 926377.0 %TSA market share 1.4065657245463854
TSA LY 65777233 PCLN LY 916512.0 %TSAmarket share 1.393357485864448
TSA PW 77421277 PCLN PW 1164076.0 %TSAmarket share 1.5035608363835176
TSA PWLY 77384934 PCLN PWly 1168906.0 %TSAmarket share 1.5105084925187118
TSA YTD CY 65860911 PCLN YTD CY 926377.0 %TSAmarket share CY YTD  1.41
TSA YTD LY 65777233 PCLN YTD LY 916512.0 %TSA market share LY YTD  1.39


In [40]:
# df_summary=df_summary.reset_index(names=['Metric'])

df_summary= pd.DataFrame()
df_summary.loc['DAU','Actual']=format_number(df_dau_conversion.loc[:, 'DAU'].loc['Total'])
df_summary.loc['DAU','Reporting Week']=df_dau_conversion.loc[:, 'DAU YoY'].loc['Total']
df_summary.loc['DAU','Previous Week']=df_dau_conversion.loc[:, 'DAU YoY_PW'].loc['Total']
df_summary.loc['DAU','YTD']=df_dau_conversion.loc[:, 'DAU YoY_YTD'].loc['Total']


df_summary.loc['TSA','Actual']=format_percentage_2(tsa_actual_cy)
df_summary.loc['TSA','Reporting Week']=format_percentage(((tsa_actual_cy/tsa_actual_cwly)-1)*100)
df_summary.loc['TSA','Previous Week']=format_percentage((tsa_actual_pw/tsa_actual_pwly-1)*100)
df_summary.loc['TSA','YTD']=format_percentage((tsa_actual_ytd/tsa_actual_ytd_ly-1)*100)

df_summary=df_summary.fillna('')

df_summary

,Actual,Reporting Week,Previous Week,YTD
DAU,8.4M,-3.0%,3.2%,-3.0%
TSA,1.41%,0.9%,-0.5%,1.4%


In [23]:
final_summary = pd.concat([finance_output,df_summary])
final_summary = final_summary.join(df_vs_plan, how="left").fillna('')
final_summary

,Actual,Reporting Week,Previous Week,YTD,Reporting Week (vs Plan),Previous Week (vs Plan),YTD (vs Plan)
Net Tickets,832K,1.1%,-2.1%,1.1%,0.1%,-0.1%,0.1%
Gross Tickets,894K,0.9%,-1.7%,0.9%,1.3%,0.1%,1.3%
Net Cont + Fee + Incentives + vcc rebate(Flight Only),8.4M,-11.7%,-2.0%,-11.7%,0.1%,-5.6%,0.1%
Gross Cont + Fee + Incentives + vcc rebate(Flight Only),9.1M,-13.0%,-1.8%,-13.0%,,,
Normalized Net Tickets,718K,1.1%,-2.5%,1.1%,,,
Normalized Gross Tickets,770K,0.4%,-2.3%,0.4%,,,
DAU,8.4M,-3.0%,3.2%,-3.0%,,,
TSA,1.41%,0.9%,-0.5%,0.0%,,,


In [41]:

import config_table
import importlib

importlib.reload(config_table)
from docx import Document
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from docx.shared import Pt

from config_table import (
    clear_document,
    set_font,
    create_word_table,
    create_summary_table,
    create_others_table,
    # create_dau_table,
    # create_roi_table
)


# Data preparation and configurations
carrier_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/Retail.jpg', 'title': 'Retail'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/Express.jpg', 'title': 'Express Deals'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

business_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/Standalone.jpg', 'title': 'Standalone'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/Package.jpg', 'title': 'Package'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

channel_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/App.jpg', 'title': 'App'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/MWeb Desktop.jpg', 'title': 'Web'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

source_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/Published.jpg', 'title': 'Published'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/Private.jpg', 'title': 'Private'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

def customize_bullet_style(paragraph, font_size):
    """
    Customize bullet style in a paragraph using XML manipulation.
    """
    # Access the paragraph's properties
    pPr = paragraph._element.get_or_add_pPr()
    numPr = OxmlElement('w:numPr')
    ilvl = OxmlElement('w:ilvl')
    ilvl.set(qn('w:val'), '0')  # Set indentation level
    numId = OxmlElement('w:numId')
    numId.set(qn('w:val'), '1')  # Set numbering ID

    numPr.append(ilvl)
    numPr.append(numId)
    pPr.append(numPr)

    # Modify font properties
    for run in paragraph.runs:
        run.font.name = "Montserrat"
        run.font.size = font_size

# Create a Word Document
word_document = Document()

# Clear the document (if necessary)
clear_document(word_document)

# Add Title
word_document.add_paragraph()
word_document.add_paragraph(f'Summary')
set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(18), bold=True)
word_document.add_paragraph('\n')

# Create Summary Table
create_summary_table(word_document,final_summary)

# Add General Notes
summary_notes = [
    'All Priceline tickets (includes B2B, Package, Express Deals, Phone Sales)',
    'Normalized tickets are counted the same as tickets, except split tickets count as 1 instead of 2',
    'Refunds are assigned to refund date',
    'Revenue is contribution w/fee + GDS incentives + VCC rebates.  Does not include package or phone sales.',
    'Daily Active Users: engaged customers in GA4',
    'TSA footprint: travel date; numerator counts all slices (OW is 1 slice; RT is 2 slices) where the first segment is either US domestic or US outbound (Priceline-only); denominator includes all people passing through TSA screening machines'
]
for note in summary_notes:
    word_document.add_paragraph(note, style='List Bullet')
    set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(7), bold=False)
    customize_bullet_style(word_document.paragraphs[-1], font_size=Pt(7))

# Add Tables for Business, Carrier, Channel, and Source
tables = [
    (df_business, "Business", business_logos, [
        'All Priceline tickets (includes Express Deals and Phone Sales, not normalized)'
    ]),
    (df_carrier, "Carrier", carrier_logos, [
        'Priceline (including phone sales), B2C, Standalone (not normalized)'
    ]),
    (df_channel, "Channel", channel_logos, [
        'Priceline (including phone sales), B2C, Standalone (includes Express Deals, not normalized)'
    ])
]
for df, title, logos, notes in tables:
    if df is None:
        continue

   # Add specific custom notes for each table
    if title == "Business":
        custom_note = "Total Business - Detail"
    elif title == "Carrier":
        custom_note = "Priceline B2C Standalone - Detail"
    else:
        custom_note = " "

    # Add the custom note before the table
    word_document.add_paragraph(custom_note)
    set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(18), bold=True)
    word_document.add_paragraph()

    create_word_table(df, title, logos, word_document)

    # word_document.add_paragraph()

    for note in notes:
        word_document.add_paragraph(note, style='List Bullet')
        # word_document.add_paragraph('\n')
        set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(7), bold=False)


# Save the Document
output_filename = os.path.join('../montly_output/', f'Flight Performance Month Ending {start_date}.docx')
word_document.save(output_filename)
print(f"Word document '{output_filename}' has been created successfully.")

# Save PDF to Shared Drive
share_drive_path = '../../../Flight Weekly Note Output/'
word_document.save(share_drive_path + f'Flight Performance Month Ending {start_date}.pdf')

print(f"Word document saved to shared drive at '{share_drive_path}' has been created successfully.")


Word document '../montly_output/Flight Performance Month Ending 2026-01-01.docx' has been created successfully.
Word document saved to shared drive at '../../../Flight Weekly Note Output/' has been created successfully.
